# AsVectorSpace

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/geometry/doc/AsVectorSpace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Note: AI was used in the creation of this example.

[`AsVectorSpace<Class>`](https://github.com/borglab/gtsam/blob/develop/gtsam/geometry/AsVectorSpace.h) is a small C++ adapter used by the cumulative spline components. It lets a manifold type, such as a camera calibration, participate in a spline when that type has local coordinates but no natural addition or group composition.

Use it only when treating the adapted value as locally affine is an intentional modeling choice. For the surrounding spline APIs, see the [CumulativeSplineTrajectory guide](../../basis/doc/CumulativeSplineTrajectory.ipynb), the [Pose2 trajectory example](../../../python/gtsam/examples/CumulativeSplineTrajectoryExample.ipynb), and the [CardinalSplineBasis example](../../../python/gtsam/examples/CardinalSplineBasisExample.ipynb).

Primary contributor: [Brett Downing](https://github.com/BrettRD).

GTSAM Copyright 2010-2022, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

:::{toc} Contents
:context: page
:depth: 2
:::

In [1]:
# Install GTSAM from pip if running in Google Colab
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass  # Not in Colab

In [2]:
import gtsam

## Why the spline needs this adapter

`CumulativeSplineTrajectory<T>` forms differences between neighboring controls, scales those differences, and accumulates them. Poses and rotations already provide the required Lie-group operations. A class such as `Cal3_S2` is only a manifold: it provides local coordinates and retraction, but adding two calibrations has no natural physical meaning.

`AsVectorSpace<Cal3_S2>` explicitly opts into an additive interpretation so the calibration can be used as a spline component. A pose and an adapted calibration can then be combined with `ProductLieGroup` when both must vary along one trajectory.

## Coordinate convention

Let $e$ be the default-constructed value of `Class`, and let $\phi(x)=\operatorname{Local}(e,x)$. The adapter defines its vector representation relative to that fixed origin. In particular,

$$x + v = \operatorname{Retract}(x,v), \qquad x + y = \operatorname{Retract}(x,\phi(y)).$$

Subtraction and negation use the corresponding negated coordinates. The `vector()` method returns $\phi(x)$.

> **Modeling constraint:** this is a chosen local affine approximation, not a newly discovered physical composition law. Use it only over a range where the default-origin coordinates are meaningful.

## A small coordinate check

`AsVectorSpace` itself is a C++ template, but the underlying convention can be seen with any GTSAM manifold. This check expresses a calibration relative to the default `Cal3_S2()` origin and retracts it again.

In [3]:
origin = gtsam.Cal3_S2()
calibration = gtsam.Cal3_S2(500.0, 510.0, 0.0, 320.0, 240.0)
coordinates = origin.localCoordinates(calibration)
reconstructed = origin.retract(coordinates)

assert reconstructed.equals(calibration, 1e-9)
coordinates

array([499., 509.,   0., 320., 240.])

## C++ usage

Define the adapted manifold type explicitly, then use it wherever the spline needs additive operations:

```cpp
using Calibration = AsVectorSpace<Cal3_S2>;

Calibration first(Cal3_S2(500.0, 505.0, 0.0, 320.0, 240.0));
Calibration second(Cal3_S2(650.0, 660.0, 0.0, 320.0, 240.0));
Calibration midpoint = first + 0.5 * (second - first).vector();
```

For a state containing both pose and calibration:

```cpp
using CameraState = ProductLieGroup<Pose3, AsVectorSpace<Cal3_S2>>;
CumulativeSplineTrajectory<CameraState> trajectory;
```

The type name keeps the approximation visible at the point where the camera state is defined.

## When to use it

Use `AsVectorSpace<Class>` when:

- `Class` has meaningful `Local` and `Retract` operations;
- a cumulative spline needs the value as a control-point component;
- the values remain in a useful neighborhood of the default origin; and
- the local affine interpretation matches the application.

Do not use it when the type already has a meaningful Lie-group composition, or when the values travel too far for one fixed coordinate chart to be appropriate.

## Source and examples

- [`AsVectorSpace.h`](https://github.com/borglab/gtsam/blob/develop/gtsam/geometry/AsVectorSpace.h)
- [`ProductLieGroup.h`](https://github.com/borglab/gtsam/blob/develop/gtsam/base/ProductLieGroup.h)
- [ProductLieGroup guide](../../base/doc/ProductLieGroup.md)
- [CumulativeSplineTrajectory guide](../../basis/doc/CumulativeSplineTrajectory.ipynb)
- [CumulativeSplineTrajectory Pose2 example](../../../python/gtsam/examples/CumulativeSplineTrajectoryExample.ipynb)
- [CardinalSplineBasis scalar example](../../../python/gtsam/examples/CardinalSplineBasisExample.ipynb)